# SmartTea-LK Component 04: Tier 1 model training

This notebook trains the two **Tier 1 prediction models** that the NSGA-II optimizer and the Action-Window scheduler use:

| Model | Predicts | Used for |
|---|---|---|
| **Yield surrogate** | green leaf, kg/ha/day, for a plucking round | NSGA-II objective: maximise yield |
| **Runoff model** | % of applied fertilizer lost to runoff | Action Window: pick a time when the forecast rain will not wash the fertilizer away |

**Evaluation protocol (the same for every model, no peeking at the test year):**
1. **Live features only**: what is known when the decision is made (forecast rain, previous-day weather, last week's sensor readings). Hindsight columns (actual rain after the application, same-day weather) are used only as an upper bound.
2. Time split: **train 2021–2023**, **validation 2024**, **test 2025**.
3. Settings are chosen **on validation 2024 only**. Boosting models early-stop on validation.
4. The chosen setting is refit on 2021–2024, and **2025 is scored once**.
5. The hardest realistic score: **unseen blocks AND future year** (5 block-grouped folds; train on the other blocks' 2021–23 rows, predict the held-out blocks' 2025 rows).

The code shares the model factories with `src/models/compare_models.py`, so this notebook and the script give the same models.
Data: `data/model_ready/` (hybrid: real weather, forecasts and soil; simulated management and crop response).

In [3]:
import os, json, time, warnings
from pathlib import Path

if Path.cwd().name == "notebooks":   # run from the project root so src/ and data/ resolve
    os.chdir("..")

warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.data.load import load_spec, load_task, monotone_constraints
from src.models.compare_models import MODELS, fit_es

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
BLUE, ORANGE, GREY = "#2a78d6", "#e08a1e", "#898781"

SAVE_MODELS = True            # write the final models to artifacts/models/
RUN_FULL_COMPARISON = False   # True = re-run all 7 models (~10 min); False = load the saved comparison table
print("project root:", Path.cwd())

ModuleNotFoundError: No module named 'joblib'

## 1. Load the model-ready data

In [ ]:
spec = load_spec()
data = {}
for task in ["yield", "runoff"]:
    X, y, m = load_task(task, "live")
    data[task] = (X, y, m)
    print(f"{task:6s}: {len(X):,} rows x {X.shape[1]} live features | target = {spec[task]['target']}")
    print("        rows per split:", m.split.value_counts().reindex(["train", "val", "test"]).to_dict(),
          "| blocks:", m.block_id.nunique(), "| estates:", m.estate_id.nunique())

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
for ax, task, unit in zip(axs, ["yield", "runoff"], ["green leaf, kg/ha/day", "runoff loss, % of applied"]):
    X, y, m = data[task]
    for s, c in [("train", BLUE), ("val", ORANGE), ("test", GREY)]:
        ax.hist(y[m.split == s], bins=40, alpha=0.55, color=c, label=s)
    ax.set_title(f"{task.capitalize()} target by split"); ax.set_xlabel(unit); ax.legend()
plt.tight_layout(); plt.show()

### Live vs hindsight features
The **hindsight** columns are known only after the event, so they are excluded from the live models:

In [ ]:
for task in ["yield", "runoff"]:
    fs = spec[task]["feature_sets"]
    print(f"{task}: hindsight-only columns ->", sorted(set(fs["hindsight"]) - set(fs["live"])))
print("\nrunoff live features:", list(data["runoff"][0].columns))

## 2. Training helpers
`train_model` follows the protocol above. `unseen_future` gives the block-grouped "unseen blocks + future year" score.

In [ ]:
def split_masks(m):
    return [(m.split == s).to_numpy() for s in ("train", "val", "test")]


def metrics(y_true, y_pred):
    return dict(r2=r2_score(y_true, y_pred), mae=mean_absolute_error(y_true, y_pred),
                rmse=mean_squared_error(y_true, y_pred) ** 0.5)


def train_model(task, name, feature_set="live"):
    """Choose settings on val 2024, refit on train+val, score test 2025 once."""
    X, y, m = data[task] if feature_set == "live" else load_task(task, feature_set)
    Xn, yn = X.to_numpy(np.float32), y.to_numpy()
    tr, va, te = split_masks(m)
    make, _, can_mono, grid = MODELS[name]
    mono = monotone_constraints(task, list(X.columns)) if (task == "runoff" and can_mono) else None
    trials = []
    for p in grid:                                   # 1. choose on val
        t0 = time.time()
        est, n = fit_es(name, make(p, None, mono), Xn[tr], yn[tr], Xn[va], yn[va])
        trials.append(dict(settings=p, iterations=n, train_r2=r2_score(yn[tr], est.predict(Xn[tr])),
                           val_r2=r2_score(yn[va], est.predict(Xn[va])), seconds=round(time.time() - t0, 1)))
    best = max(trials, key=lambda t: t["val_r2"])
    final = make(best["settings"], best["iterations"], mono).fit(Xn[tr | va], yn[tr | va])   # 2. refit
    pred = final.predict(Xn[te])                     # 3. test once
    return dict(task=task, name=name, feature_set=feature_set, model=final, X=X, y=y, m=m, mono=mono,
                trials=pd.DataFrame(trials), best=best, pred_test=pred, test=metrics(yn[te], pred))


def unseen_future(r):
    """Train on other blocks' 2021-23 rows, predict the held-out blocks' 2025 rows (5 folds)."""
    X, y, m = r["X"], r["y"], r["m"]
    Xn, yn = X.to_numpy(np.float32), y.to_numpy()
    tr, _, te = split_masks(m)
    make = MODELS[r["name"]][0]
    out = []
    for k in range(5):
        a, b = (m.cv_fold != k).to_numpy() & tr, (m.cv_fold == k).to_numpy() & te
        est = make(r["best"]["settings"], r["best"]["iterations"], r["mono"]).fit(Xn[a], yn[a])
        out.append(r2_score(yn[b], est.predict(Xn[b])))
    return np.array(out)


def report(r, cv):
    b, t = r["best"], r["test"]
    print(f"{r['task']} / {r['name']} ({r['feature_set']} features)")
    print(f"  chosen settings : {b['settings']}  iterations: {b['iterations']}")
    print(f"  train R2 {b['train_r2']:.3f} | val 2024 R2 {b['val_r2']:.3f} | test 2025 R2 {t['r2']:.3f} "
          f"(MAE {t['mae']:.2f}, RMSE {t['rmse']:.2f})")
    print(f"  unseen blocks + future year R2: {cv.mean():.3f} +/- {cv.std():.3f}  (folds: {np.round(cv, 3)})")
    print(f"  train - val gap: {b['train_r2'] - b['val_r2']:.3f}")


def plot_pred(r, unit):
    m, te = r["m"], split_masks(r["m"])[2]
    yt, yp = r["y"][te].to_numpy(), r["pred_test"]
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.2]})
    axs[0].scatter(yt, yp, s=6, alpha=0.35, color=BLUE, lw=0)
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    axs[0].plot([lo, hi], [lo, hi], color=GREY, lw=1, ls="--")
    axs[0].set_xlabel(f"actual ({unit})"); axs[0].set_ylabel("predicted")
    axs[0].set_title(f"Test 2025: predicted vs actual (R2 {r['test']['r2']:.3f})")
    est = pd.DataFrame({"e": m.estate_id[te].to_numpy(), "y": yt, "p": yp}).groupby("e").apply(
        lambda g: r2_score(g.y, g.p) if len(g) > 2 else np.nan)
    axs[1].bar(est.index, est.values, color=BLUE, width=0.6)
    for i, v in enumerate(est.values):
        axs[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    axs[1].set_title("Test 2025 R2 per estate"); axs[1].set_ylim(min(0, est.min() - 0.1), 1)
    plt.tight_layout(); plt.show()

## 3. Baselines
A model must beat these to be useful: predicting the training mean (R² ≈ 0) and a linear Ridge regression.

In [ ]:
rows = []
for task in ["yield", "runoff"]:
    X, y, m = data[task]
    tr, va, te = split_masks(m)
    rows.append(dict(task=task, model="train mean", test_r2=r2_score(y[te], np.full(te.sum(), y[tr | va].mean()))))
    rr = train_model(task, "Ridge")
    rows.append(dict(task=task, model=f"Ridge {rr['best']['settings']}", test_r2=rr["test"]["r2"]))
pd.DataFrame(rows).round(3)

## 4. Yield surrogate: CatBoost
CatBoost had the highest validation R² of the 7 models (see section 7). Candidate settings, each early-stopped on 2024:

In [ ]:
res_yield = train_model("yield", "CatBoost")
display(res_yield["trials"].round(3))
cv_yield = unseen_future(res_yield)
report(res_yield, cv_yield)

In [ ]:
plot_pred(res_yield, "kg/ha/day")

## 5. Runoff model: XGBoost with monotone rain constraints
More forecast rain after an application must **never** reduce the predicted runoff, so the forecast-rain features
have monotone (+1) constraints. Temperature and humidity are the **previous day's** values: the same-day daily means
partly contain that day's rain (the humidity leak fixed on 2026-10-08).

In [ ]:
res_runoff = train_model("runoff", "XGBoost")
display(res_runoff["trials"].round(3))
cv_runoff = unseen_future(res_runoff)
report(res_runoff, cv_runoff)

In [ ]:
plot_pred(res_runoff, "% lost")

Per-estate R² uses only 30–40 test applications per estate, so it is noisy. The **up-country estates are harder**:
test MAE is about 14 percentage points for Nuwara Eliya (E01) and Hatton (E02), against about 8 for Kandy, Ratnapura and
Deniyaya. E01 is the weakest (R² below 0): its errors are as large as E02's while its runoff values spread less.
Report this as a limitation for up-country blocks.

### Monotonic check: add 10 mm to every forecast-rain column; predicted runoff must not go down

In [ ]:
X, y, m = res_runoff["X"], res_runoff["y"], res_runoff["m"]
te = split_masks(m)[2]
rain_cols = [c for c in X.columns if c.startswith("fc_rain")]
base = res_runoff["model"].predict(X[te].to_numpy(np.float32))
up = X[te].copy(); up[rain_cols] += 10
diff = res_runoff["model"].predict(up.to_numpy(np.float32)) - base
print(f"rows tested: {te.sum()} | rows where runoff went DOWN: {(diff < -1e-6).sum()} | "
      f"mean increase: {diff.mean():.2f} percentage points")

### Forecast vs perfect knowledge (upper bound)
The same model trained on the **actual** rain after the application shows how much the forecast error costs.
This is the key thesis finding: forecast error, not the model, limits the Action Window.

In [ ]:
res_hind = train_model("runoff", "XGBoost", feature_set="hindsight")
cv_hind = unseen_future(res_hind)
pd.DataFrame([
    dict(features="live (forecast rain, previous-day weather)", test_r2=res_runoff["test"]["r2"],
         test_mae=res_runoff["test"]["mae"], unseen_future_r2=cv_runoff.mean()),
    dict(features="hindsight (actual rain, same-day weather) - upper bound", test_r2=res_hind["test"]["r2"],
         test_mae=res_hind["test"]["mae"], unseen_future_r2=cv_hind.mean()),
]).round(3)

### Runoff risk: high or low?
The Action Window does not need the exact loss. It needs to know whether applying now is **high or low risk**.
**High risk** = more than the median loss of the training years 2021–2023. The cut-off is fixed on the training data
before scoring and is not tuned. The same model is used; its predicted loss is compared with the cut-off.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

y_tr = res_runoff["y"][res_runoff["m"].split == "train"]
RISK_CUT = float(np.median(y_tr))
te = split_masks(res_runoff["m"])[2]
actual_high = res_runoff["y"][te].to_numpy() > RISK_CUT
pred_high = res_runoff["pred_test"] > RISK_CUT


def risk_unseen_future(r, cut):
    """High/low accuracy for unseen blocks AND future year (same 5 folds as the R2 score)."""
    X, y, m = r["X"], r["y"], r["m"]
    Xn, yn = X.to_numpy(np.float32), y.to_numpy()
    tr, _, te_ = split_masks(m)
    make, out = MODELS[r["name"]][0], []
    for k in range(5):
        a, b = (m.cv_fold != k).to_numpy() & tr, (m.cv_fold == k).to_numpy() & te_
        est = make(r["best"]["settings"], r["best"]["iterations"], r["mono"]).fit(Xn[a], yn[a])
        out.append(accuracy_score(yn[b] > cut, est.predict(Xn[b]) > cut))
    return np.array(out)


risk_cv = risk_unseen_future(res_runoff, RISK_CUT)
risk = dict(cut_pct=RISK_CUT, test_accuracy=accuracy_score(actual_high, pred_high),
            test_balanced_accuracy=balanced_accuracy_score(actual_high, pred_high),
            majority_baseline=max(actual_high.mean(), 1 - actual_high.mean()),
            unseen_future_accuracy=risk_cv.mean(), unseen_future_sd=risk_cv.std())
print(f"high risk = loss above {RISK_CUT:.1f}% (median of 2021-2023)")
print(f"test 2025 accuracy {risk['test_accuracy']:.3f} (balanced {risk['test_balanced_accuracy']:.3f}) | "
      f"always guessing the larger class: {risk['majority_baseline']:.3f}")
print(f"unseen blocks + future year accuracy: {risk_cv.mean():.3f} +/- {risk_cv.std():.3f}")

cm = confusion_matrix(actual_high, pred_high)
fig, ax = plt.subplots(figsize=(3.6, 3.2))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=12, color="white" if v > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["low", "high"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["low", "high"])
ax.set_xlabel("predicted risk"); ax.set_ylabel("actual risk")
ax.set_title(f"Test 2025: {cm.trace()} of {cm.sum()} correct", fontsize=9)
plt.tight_layout(); plt.show()

## 6. Overfitting checks
### Train / validation / test / unseen gap

In [ ]:
summary = pd.DataFrame([
    dict(task=r["task"], model=r["name"], train_r2=r["best"]["train_r2"], val_r2=r["best"]["val_r2"],
         test_r2=r["test"]["r2"], test_mae=r["test"]["mae"], unseen_future_r2=cv.mean(), unseen_future_sd=cv.std())
    for r, cv in [(res_yield, cv_yield), (res_runoff, cv_runoff)]])
display(summary.round(3))

fig, ax = plt.subplots(figsize=(8, 3))
cols = ["train_r2", "val_r2", "test_r2", "unseen_future_r2"]
w = 0.2
for i, c in enumerate(cols):
    ax.bar(np.arange(2) + (i - 1.5) * w, summary[c], width=w, label=c.replace("_r2", "").replace("_", " + "),
           color=[BLUE, ORANGE, GREY, "#1c5cab"][i])
ax.set_xticks(range(2)); ax.set_xticklabels(summary.task); ax.set_ylim(0, 1); ax.set_ylabel("R2")
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False); ax.set_title("R2 from training data to the hardest unseen test")
plt.tight_layout(); plt.show()

### Learning curves
Each model is trained on a growing share of the 2021–2023 rows with its chosen settings and scored on 2024.
If the validation curve is still rising at 100 %, more data would help. A large, constant gap means overfitting.

In [ ]:
def learning_curve(r, fractions=(0.2, 0.4, 0.6, 0.8, 1.0), seed=0):
    X, y, m = r["X"], r["y"], r["m"]
    Xn, yn = X.to_numpy(np.float32), y.to_numpy()
    tr, va, _ = split_masks(m)
    idx = np.random.default_rng(seed).permutation(np.where(tr)[0])
    make, out = MODELS[r["name"]][0], []
    for f in fractions:
        sub = idx[: int(len(idx) * f)]
        est = make(r["best"]["settings"], r["best"]["iterations"], r["mono"]).fit(Xn[sub], yn[sub])
        out.append(dict(fraction=f, rows=len(sub), train_r2=r2_score(yn[sub], est.predict(Xn[sub])),
                        val_r2=r2_score(yn[va], est.predict(Xn[va]))))
    return pd.DataFrame(out)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, r in zip(axs, [res_yield, res_runoff]):
    lc = learning_curve(r)
    ax.plot(lc.rows, lc.train_r2, "o-", color=BLUE, label="train")
    ax.plot(lc.rows, lc.val_r2, "o-", color=ORANGE, label="validation 2024")
    ax.set_title(f"{r['task'].capitalize()} ({r['name']})"); ax.set_xlabel("training rows"); ax.set_ylabel("R2")
    ax.set_ylim(0, 1.02); ax.legend()
    print(r["task"]); display(lc.round(3))
plt.tight_layout(); plt.show()

**Reading the curves:** yield validation R² levels off at about 0.83 from 60 % of the rows, so more rows of the same kind
add little. Runoff validation R² is still rising slowly (0.64 -> 0.69), so more fertilizer applications would help it.
The training-validation gap shrinks as data grows, which is the expected pattern for a regularised model.

## 7. Comparison with the other 6 models
All 7 models (Ridge, Random Forest, HistGradientBoosting, XGBoost, LightGBM, CatBoost, MLP) use exactly the same protocol.
The winner per task is picked by **validation R² only**.

In [ ]:
if RUN_FULL_COMPARISON:
    from src.models import compare_models
    compare_models.main()
cmp = pd.read_csv("artifacts/reports/model_comparison.csv")
for task, g in cmp.groupby("task", sort=False):
    print(f"== {task} ==")
    display(g[["model", "train_r2", "val_r2", "test_r2", "test_mae", "unseen_future_r2", "unseen_future_sd",
               "predict_ms_per_10k", "best_by_val"]].sort_values("val_r2", ascending=False).round(3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (task, g) in zip(axs, cmp.groupby("task", sort=False)):
    g = g.sort_values("val_r2")
    yy = np.arange(len(g))
    ax.barh(yy + 0.2, g.test_r2.clip(lower=-0.2), height=0.38, color=BLUE, label="test 2025")
    ax.barh(yy - 0.2, g.unseen_future_r2.clip(lower=-0.2), height=0.38, color=ORANGE, label="unseen blocks + future")
    ax.set_yticks(yy); ax.set_yticklabels(g.model); ax.set_xlim(-0.2, 1); ax.set_xlabel("R2 (clipped at -0.2)")
    ax.set_title(f"{task.capitalize()}: models sorted by val R2"); ax.legend(fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.16), frameon=False)
plt.tight_layout(); plt.show()

## 8. What drives the predictions (SHAP)
Mean |SHAP| per feature, with one-hot columns summed back into their parent feature.
Full charts and the agronomy direction check: `python -m src.models.explain_shap` -> `artifacts/reports/shap/`.

In [ ]:
def shap_importance(r, top=12):
    X = r["X"]
    sv = shap.TreeExplainer(r["model"]).shap_values(X.to_numpy(np.float32))
    groups = pd.Index([c.split("__")[0] for c in X.columns])
    imp = pd.DataFrame(np.abs(sv), columns=X.columns).T.groupby(groups, sort=False).sum().T.mean()
    imp = imp.sort_values(ascending=False)
    return (imp / imp.sum() * 100).head(top)

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, r in zip(axs, [res_yield, res_runoff]):
    s = shap_importance(r).iloc[::-1]
    ax.barh(s.index, s.values, color=BLUE, height=0.6)
    for i, v in enumerate(s.values):
        ax.text(v, i, f" {v:.0f}%", va="center", fontsize=8)
    ax.set_xlabel("share of mean |SHAP| (%)"); ax.set_title(f"{r['task'].capitalize()} ({r['name']}): top drivers")
plt.tight_layout(); plt.show()

**Reading the drivers**
- **Yield:** soil N in the week before the round, cultivar, days since pruning and elevation zone. Fungicide appears to
  reduce yield only because of confounding: sprays happen in wet, cloudy blight weather. NSGA-II must **not** choose
  fungicide through this model; fungicide is decided by a blight-risk rule plus the Action Window.
- **Runoff:** soil moisture at application, rain in the previous 3 days, forecast rain and slope. These are the
  physically expected drivers.

## 9. Save the final models

In [ ]:
def save(r, cv, extra=None):
    stem = f"{r['task']}_{r['name'].lower().replace(' ', '_')}"
    meta = dict(task=r["task"], model=r["name"], settings=r["best"]["settings"], iterations=r["best"]["iterations"],
                features=list(r["X"].columns), feature_set=r["feature_set"], target=spec[r["task"]]["target"],
                trained_on="train+val (2021-2024)", monotone=bool(r["mono"]),
                scores=dict(val_r2=round(r["best"]["val_r2"], 4), test_r2=round(r["test"]["r2"], 4),
                            test_mae=round(r["test"]["mae"], 4), test_rmse=round(r["test"]["rmse"], 4),
                            unseen_future_r2=round(cv.mean(), 4), unseen_future_sd=round(cv.std(), 4)),
                selected_by="highest val R2 (2024) among all models and settings")
    if extra:
        meta["scores"].update(extra)
    os.makedirs("artifacts/models", exist_ok=True)
    joblib.dump(r["model"], f"artifacts/models/{stem}.joblib")
    json.dump(meta, open(f"artifacts/models/{stem}.json", "w"), indent=2)
    print(f"saved artifacts/models/{stem}.joblib  (+ .json)")

if SAVE_MODELS:
    save(res_yield, cv_yield)
    save(res_runoff, cv_runoff, extra={f"risk_{k}": round(float(v), 4) for k, v in risk.items()})

# reload check: the saved model gives the same predictions
m2 = joblib.load("artifacts/models/runoff_xgboost.joblib")
te = split_masks(res_runoff["m"])[2]
print("reloaded runoff model identical:",
      np.allclose(m2.predict(res_runoff["X"][te].to_numpy(np.float32)), res_runoff["pred_test"]))

## 10. Summary

| Task | Model | Metric | Test 2025 | Unseen blocks + future year |
|---|---|---|---|---|
| Yield | CatBoost | R² (variation explained) | **0.91** (MAE 1.63 kg/ha/day) | **0.80 ± 0.03** |
| Runoff risk (high / low) | XGBoost (monotone) | classification accuracy | **0.81** (guessing: 0.54) | **0.79 ± 0.08** |
| Runoff amount | XGBoost (monotone) | R² | 0.57 (MAE 9.9 pp) | 0.44 ± 0.19 |
| Runoff amount, actual rain (upper bound only) | XGBoost | R² | 0.94 | 0.90 |

- R² is not "accuracy": R² 0.91 means the model explains 91 % of the variation in yield.
- The runoff risk cut-off (median loss of 2021–2023) was fixed before scoring.
- Settings were chosen on 2024 only; 2025 was scored once. No tuning toward a target accuracy.
- Runoff is harder because it is predicted from the **forecast** rain. With the actual rain after the application
  it reaches 0.94 (section 5): forecast error is the main limit, not the model.
- New estates (new locations) are **not** predictable (leave-one-estate-out yield R² about 0.1). Report this.
- The data is a hybrid: weather, forecasts and soil are real; management, crop response and runoff are simulated.
  Validation on real estate records (TRI) is still the main open step.

**Used by:** Tier 2, the NSGA-II optimizer (`src/optimizer/`), and next the Tier 3 Action-Window scheduler.